# Lasso en inferencia causal

En este taller usaremos el estimador de Lasso para apoyar la inferencia causal. Partimos del modelo de Acemoglu, Johnson y Robinson (2001), donde se estima el efecto de las instituciones (riesgo de expropiación, `Exprop`) sobre el PIB per cápita (`GDP`), usando la mortalidad de los colonos (`lnmort`) como instrumento.

\begin{equation}
GDP=\alpha\, Exprop+\beta'W+\epsilon
\end{equation}

donde $W$ contiene controles geográficos. La pregunta es cuáles controles incluir; usaremos Lasso para seleccionarlos.

In [1]:
suppressWarnings(
pacman::p_load(dplyr, hdm, AER, sandwich, lmtest)
)

Installing package into ‘C:/Users/andre/AppData/Local/R/win-library/4.5’
(as ‘lib’ is unspecified)


trying URL 'https://cran.rstudio.com/bin/windows/contrib/4.5/AER_1.2-17.zip'
Content type 'application/zip' length 2591515 bytes (2.5 MB)
downloaded 2.5 MB



package ‘AER’ successfully unpacked and MD5 sums checked

The downloaded binary packages are in
	C:\Users\andre\AppData\Local\Temp\Rtmp4AG48y\downloaded_packages

AER installed


## Datos

Leemos el archivo `acemoglu_col_notext.txt` (separado por tabulaciones). Si no está en la carpeta de trabajo, usamos la misma base que trae el paquete `hdm` (`data(AJR)`, 64 países).

In [2]:
if (file.exists("acemoglu_col_notext.txt")) {
  df <- read.delim("acemoglu_col_notext.txt")
} else {
  data(AJR, package = "hdm")
  df <- AJR[, c("GDP", "Exprop", "Mort", "Latitude", "Neo", "Africa", "Asia", "Namer", "Samer")]
}

head(df)
dim(df)

[1] 64  9

## Creación de variables y transformaciones

Además del logaritmo de la mortalidad, construimos un polinomio cúbico de la latitud y *splines* cúbicos con nodos en 0.08, 0.16 y 0.24.

In [3]:
df <- df |>
  mutate(lnmort   = log(Mort),
         lat2     = Latitude^2,
         lat3     = Latitude^3,
         lat_c08  = pmax(Latitude - 0.08, 0),
         lat2_c08 = lat_c08^2,
         lat3_c08 = lat_c08^3,
         lat_c16  = pmax(Latitude - 0.16, 0),
         lat2_c16 = lat_c16^2,
         lat3_c16 = lat_c16^3,
         lat_c24  = pmax(Latitude - 0.24, 0),
         lat2_c24 = lat_c24^2,
         lat3_c24 = lat_c24^3)

head(df)
dim(df)

[1] 64 21

## Regresión IV base

$GDP$ (dependiente) ~ $Exprop$ (endógena) | $lnmort$ (instrumento), controlando por latitud. En `ivreg` la fórmula es `y ~ regresores | instrumentos`, donde los controles exógenos aparecen a ambos lados de la barra.

Usamos errores robustos HC0, que corresponden a `cov_type='heteroskedastic'` de `linearmodels` en Python.

In [4]:
iv_base <- ivreg(GDP ~ Exprop + Latitude | lnmort + Latitude, data = df)
coeftest(iv_base, vcov = vcovHC(iv_base, type = "HC0"))


t test of coefficients:

            Estimate Std. Error t value  Pr(>|t|)    
(Intercept)  1.87440    1.28346  1.4604    0.1493    
Exprop       0.96924    0.20778  4.6648 1.734e-05 ***
Latitude    -0.66961    0.87747 -0.7631    0.4483    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


### Primera etapa

In [5]:
fs_base <- lm(Exprop ~ Latitude + lnmort, data = df)
coeftest(fs_base, vcov = vcovHC(fs_base, type = "HC1"))


t test of coefficients:

            Estimate Std. Error t value  Pr(>|t|)    
(Intercept)  8.66596    0.80711 10.7370 1.094e-15 ***
Latitude     1.81774    1.06158  1.7123 0.0919210 .  
lnmort      -0.53717    0.15449 -3.4771 0.0009405 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


## Incluyendo todos los controles

Con 16 controles y solo 64 observaciones, la estimación pierde precisión.

In [6]:
controls <- c("Africa", "Asia", "Namer", "Samer", "Latitude", "lat2", "lat3",
              "lat_c08", "lat2_c08", "lat3_c08", "lat_c16", "lat2_c16", "lat3_c16",
              "lat_c24", "lat2_c24", "lat3_c24")

ctrl_txt <- paste(controls, collapse = " + ")
ctrl_txt

[1] "Africa + Asia + Namer + Samer + Latitude + lat2 + lat3 + lat_c08 + lat2_c08 + lat3_c08 + lat_c16 + lat2_c16 + lat3_c16 + lat_c24 + lat2_c24 + lat3_c24"

In [7]:
iv_full <- ivreg(as.formula(paste("GDP ~ Exprop +", ctrl_txt, "| lnmort +", ctrl_txt)), data = df)
coeftest(iv_full, vcov = vcovHC(iv_full, type = "HC0"))


t test of coefficients:

               Estimate  Std. Error t value Pr(>|t|)
(Intercept)  3.1153e+00  3.6459e+00  0.8545   0.3973
Exprop       9.8906e-01  6.7862e-01  1.4575   0.1518
Africa       5.4516e-02  1.2892e+00  0.0423   0.9665
Asia        -2.2193e-01  8.5367e-01 -0.2600   0.7960
Namer        6.6263e-01  9.5167e-01  0.6963   0.4898
Samer        7.5865e-01  1.0708e+00  0.7085   0.4822
Latitude    -1.3808e+02  1.9672e+02 -0.7019   0.4863
lat2         2.5435e+03  4.9025e+03  0.5188   0.6064
lat3        -1.4792e+04  3.4547e+04 -0.4282   0.6705
lat_c08      3.6508e+01  1.6453e+02  0.2219   0.8254
lat2_c08     5.1347e+02  3.3104e+03  0.1551   0.8774
lat3_c08     2.0052e+04  5.1148e+04  0.3921   0.6968
lat_c16     -3.4573e+01  1.7588e+02 -0.1966   0.8450
lat2_c16    -1.4133e+03  3.0276e+03 -0.4668   0.6428
lat3_c16    -5.7702e+02  4.0821e+04 -0.0141   0.9888
lat_c24      2.1674e+01  1.1349e+02  0.1910   0.8494
lat2_c24    -6.2588e+02  2.7988e+03 -0.2236   0.8240
lat3_c24    -4.4465e

### Primera etapa

In [8]:
fs_full <- lm(as.formula(paste("Exprop ~ lnmort +", ctrl_txt)), data = df)
coeftest(fs_full, vcov = vcovHC(fs_full, type = "HC1"))


t test of coefficients:

               Estimate  Std. Error t value  Pr(>|t|)    
(Intercept)  5.9792e+00  1.4405e+00  4.1507 0.0001417 ***
lnmort      -2.1818e-01  2.2206e-01 -0.9825 0.3309873    
Africa      -1.0493e+00  1.1997e+00 -0.8746 0.3863084    
Asia        -3.3620e-01  1.2147e+00 -0.2768 0.7831917    
Namer       -6.0618e-01  1.1695e+00 -0.5183 0.6067169    
Samer       -9.1258e-01  1.1108e+00 -0.8215 0.4155748    
Latitude     2.5836e+02  6.5395e+01  3.9508 0.0002657 ***
lat2        -6.1168e+03  2.2330e+03 -2.7392 0.0087331 ** 
lat3         4.0830e+04  1.9391e+04  2.1056 0.0407304 *  
lat_c08     -5.5389e+01  1.8554e+02 -0.2985 0.7666481    
lat2_c08    -3.5429e+03  2.9185e+03 -1.2140 0.2309539    
lat3_c08    -4.6828e+04  4.0186e+04 -1.1653 0.2499110    
lat_c16      1.1231e+02  1.9387e+02  0.5793 0.5651915    
lat2_c16     4.8737e+02  3.7404e+03  0.1303 0.8968978    
lat3_c16     1.6010e+04  4.3277e+04  0.3699 0.7131224    
lat_c24     -1.1122e+02  1.0905e+02 -1.0198 0.

Note que el instrumento pierde significancia en la primera etapa cuando incluimos todos los controles.

## Selección de controles con Lasso

Estimamos tres ecuaciones con `rlasso` (penalidad teórica): el resultado, la variable endógena y el instrumento, cada una contra los controles. Los controles seleccionados en **cualquiera** de las tres ecuaciones se incluyen en la regresión IV final.

In [9]:
X <- as.matrix(df[, controls])

# Ecuación 1: PIB per cápita ~ controles
lasso_gdp <- rlasso(X, df$GDP, post = FALSE)

# Ecuación 2: Exprop ~ controles
lasso_exprop <- rlasso(X, df$Exprop, post = FALSE)

# Ecuación 3: Mortalidad ~ controles
lasso_inst <- rlasso(X, df$lnmort, post = FALSE)

Coeficientes distintos de cero en cada ecuación:

In [10]:
nonzero <- function(fit) {
  b <- coef(fit)
  b[abs(b) > 1e-10]
}

cat("1. GDP ~ controles\n");    print(round(nonzero(lasso_gdp), 4))
cat("\n2. Exprop ~ controles\n"); print(round(nonzero(lasso_exprop), 4))
cat("\n3. lnmort ~ controles\n"); print(round(nonzero(lasso_inst), 4))

1. GDP ~ controles
(Intercept)      Africa     lat_c16 
     8.2970     -0.8982      2.0731 

2. Exprop ~ controles
(Intercept)      Africa     lat_c16 
     6.5664     -0.5376      2.5331 

3. lnmort ~ controles
(Intercept)      Africa     lat_c16 
     4.3612      1.0324     -2.1518 


In [11]:
selected <- function(fit) {
  b <- coef(fit)[-1]          # sin intercepto
  names(b)[abs(b) > 1e-10]
}

sel_controls <- union(union(selected(lasso_gdp), selected(lasso_exprop)), selected(lasso_inst))

cat("Ecuación GDP:   ", length(selected(lasso_gdp)), "/", length(controls), "\n")
cat("Ecuación Exprop:", length(selected(lasso_exprop)), "/", length(controls), "\n")
cat("Ecuación lnmort:", length(selected(lasso_inst)), "/", length(controls), "\n")
cat("Controles seleccionados (unión):", sel_controls, "\n")

Ecuación GDP:    2 / 16 
Ecuación Exprop: 2 / 16 
Ecuación lnmort: 2 / 16 
Controles seleccionados (unión): Africa lat_c16 


## IV con los controles seleccionados

In [12]:
sel_txt <- if (length(sel_controls) > 0) paste(sel_controls, collapse = " + ") else "1"

iv_lasso <- ivreg(as.formula(paste("GDP ~ Exprop +", sel_txt, "| lnmort +", sel_txt)), data = df)
coeftest(iv_lasso, vcov = vcovHC(iv_lasso, type = "HC0"))


t test of coefficients:

            Estimate Std. Error t value Pr(>|t|)  
(Intercept)  2.81903    2.15591  1.3076  0.19600  
Exprop       0.83492    0.32453  2.5727  0.01258 *
Africa      -0.34421    0.39037 -0.8817  0.38143  
lat_c16     -0.74204    1.62238 -0.4574  0.64905  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


### Primera etapa

In [13]:
fs_lasso <- lm(as.formula(paste("Exprop ~ lnmort +", sel_txt)), data = df)
coeftest(fs_lasso, vcov = vcovHC(fs_lasso, type = "HC1"))


t test of coefficients:

            Estimate Std. Error t value  Pr(>|t|)    
(Intercept)  8.23170    0.80667 10.2046 9.875e-15 ***
lnmort      -0.38024    0.17761 -2.1409  0.036355 *  
Africa      -0.45360    0.33564 -1.3514  0.181629    
lat_c16      3.48115    1.30638  2.6647  0.009883 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


## Comparación con `rlassoIV`

El paquete `hdm` implementa este procedimiento directamente. Con `select.X = TRUE` y `select.Z = FALSE`, Lasso selecciona los controles y el instrumento se mantiene fijo; el efecto se estima por *partialling-out*.

In [14]:
iv_hdm <- rlassoIV(x = X, d = df$Exprop, y = df$GDP, z = df$lnmort,
                   select.X = TRUE, select.Z = FALSE)
summary(iv_hdm)
confint(iv_hdm)

[1] "Estimation and significance testing of the effect of target variables in the IV regression model"
   coeff.    se. t-value p-value   
d1 0.8349 0.2985   2.797 0.00516 **
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


       2.5 %   97.5 %
d1 0.2498633 1.419971


### Resumen

In [15]:
res <- rbind(
  "IV base (latitud)"   = coeftest(iv_base,  vcov = vcovHC(iv_base,  type = "HC0"))["Exprop", 1:2],
  "IV todos controles"  = coeftest(iv_full,  vcov = vcovHC(iv_full,  type = "HC0"))["Exprop", 1:2],
  "IV controles Lasso"  = coeftest(iv_lasso, vcov = vcovHC(iv_lasso, type = "HC0"))["Exprop", 1:2],
  "rlassoIV (hdm)"      = c(iv_hdm$coefficients, iv_hdm$se)
)
colnames(res) <- c("Coef. Exprop", "EE")
round(res, 4)

                   Coef. Exprop     EE
IV base (latitud)        0.9692 0.2078
IV todos controles       0.9891 0.6786
IV controles Lasso       0.8349 0.3245
rlassoIV (hdm)           0.8349 0.2985

## DML

En esta sección estimamos el efecto de `Exprop` con *double/debiased machine learning* (DML). Como `Exprop` es endógena, usamos el modelo parcialmente lineal con variable instrumental (PLIV):

\begin{equation}
GDP=\theta\, Exprop+g(X)+\epsilon,\qquad E[\epsilon\mid Z,X]=0
\end{equation}

donde $Z=lnmort$. Con *partialling-out* residualizamos las tres variables respecto a los controles:

$$
\check Y=Y-\hat\ell(X),\qquad \check D=D-\hat m(X),\qquad \check Z=Z-\hat r(X)
$$

Luego estimamos $\theta$ por IV de $\check Y$ sobre $\check D$, con $\check Z$ como instrumento:

$$
\hat\theta=\frac{\mathbb{E}_n[\check Z\check Y]}{\mathbb{E}_n[\check Z\check D]}
$$

Las funciones $\ell$, $m$ y $r$ se estiman con *cross-fitting*:

1. **Partición.** Dividimos la muestra al azar en $K$ folds $I_1,\dots,I_K$.
2. **Entrenamiento fuera del fold.** Para cada $k$, estimamos $\hat\ell_{-k}$, $\hat m_{-k}$ y $\hat r_{-k}$ con las observaciones que no están en $I_k$.
3. **Predicción dentro del fold.** Para $i\in I_k$, calculamos los residuales $\check Y_i$, $\check D_i$ y $\check Z_i$.
4. **Rotación.** Repetimos 2–3 para $k=1,\dots,K$, de modo que cada observación tenga su residual fuera de muestra.
5. **Estimación.** Con todos los residuales, estimamos $\theta$ por IV.

Como $\hat\theta$ depende del sorteo de folds, repetimos el procedimiento $R$ veces y reportamos la mediana.

Usaremos dos métodos de ML para las funciones fastidiosas: **Lasso con validación cruzada** (`cv.glmnet`, $\lambda$ elegido por CV *dentro* de la muestra de entrenamiento de cada fold) y **bosques aleatorios** (`randomForest`).

In [16]:
suppressWarnings(
pacman::p_load(glmnet, randomForest)
)

### Métodos de ML para las funciones fastidiosas

Cada función recibe los datos de entrenamiento y devuelve las predicciones para el fold que se deja por fuera.

In [17]:
# Lasso con validación cruzada (lambda.min), CV hecha dentro de la muestra de entrenamiento
lasso_cv <- function(X_train, y_train, X_test) {
  fit <- cv.glmnet(X_train, y_train, family = "gaussian", alpha = 1, nfolds = 5,
                   lambda.min.ratio = 0.01)
  as.numeric(predict(fit, newx = X_test, s = "lambda.min"))
}

# Bosque aleatorio
rf <- function(X_train, y_train, X_test) {
  fit <- randomForest(x = X_train, y = y_train, ntree = 1000, nodesize = 5)
  as.numeric(predict(fit, newdata = X_test))
}

### Cross-fitting

La función `dml_iv` hace el *cross-fitting* para un sorteo de folds y devuelve $\hat\theta$, su error estándar y los residuales. El error estándar es el robusto (HC0) de la regresión IV sin intercepto de $\check Y$ sobre $\check D$ con instrumento $\check Z$, que coincide con la varianza del *score* ortogonal de DML.

In [18]:
dml_iv <- function(X, y, d, z, learner, K = 5) {
  n <- length(y)
  folds <- sample(rep(1:K, length.out = n))   # 1. partición aleatoria
  yhat <- dhat <- zhat <- rep(NA_real_, n)

  for (k in 1:K) {
    test  <- which(folds == k)
    train <- which(folds != k)
    # 2. entrenamiento en I_k^c y 3. predicción en I_k
    yhat[test] <- learner(X[train, , drop = FALSE], y[train], X[test, , drop = FALSE])
    dhat[test] <- learner(X[train, , drop = FALSE], d[train], X[test, , drop = FALSE])
    zhat[test] <- learner(X[train, , drop = FALSE], z[train], X[test, , drop = FALSE])
  }                                             # 4. rotación sobre los K folds

  ytil <- y - yhat
  dtil <- d - dhat
  ztil <- z - zhat

  # 5. IV de los residuales
  fit <- ivreg(ytil ~ dtil - 1 | ztil - 1)
  ct  <- coeftest(fit, vcov = vcovHC(fit, type = "HC0"))

  list(coef = ct["dtil", "Estimate"], se = ct["dtil", "Std. Error"],
       ytil = ytil, dtil = dtil, ztil = ztil)
}

Para agregar las $R$ repeticiones usamos la mediana de los $\hat\theta_r$. El error estándar agregado incluye la variabilidad entre sorteos (Chernozhukov et al., 2018):

$$
\hat\sigma=\sqrt{\text{mediana}_r\left(\hat\sigma_r^2+(\hat\theta_r-\hat\theta)^2\right)}
$$

In [19]:
dml_iv_rep <- function(X, y, d, z, learner, K = 5, R = 10, seed = 1234) {
  set.seed(seed)
  reps <- t(replicate(R, {
    r <- dml_iv(X, y, d, z, learner, K)
    c(coef = r$coef, se = r$se)
  }))
  theta <- median(reps[, "coef"])
  se    <- sqrt(median(reps[, "se"]^2 + (reps[, "coef"] - theta)^2))
  list(coef = theta, se = se, reps = reps)
}

### Estimación

Para Lasso usamos la matriz de 16 controles, `X`, que incluye el polinomio y los *splines* de latitud. El bosque aleatorio captura no linealidades por sí solo, así que basta con las variables originales: los continentes y la latitud.

In [20]:
y <- df$GDP
d <- df$Exprop
z <- df$lnmort

X_rf <- as.matrix(df[, c("Africa", "Asia", "Namer", "Samer", "Latitude")])

dml_lasso <- dml_iv_rep(X,    y, d, z, learner = lasso_cv, K = 5, R = 10)
dml_rf    <- dml_iv_rep(X_rf, y, d, z, learner = rf,       K = 5, R = 10)

cat("DML Lasso CV     | Coef. Exprop:", round(dml_lasso$coef, 4), " | EE:", round(dml_lasso$se, 4), "\n")
cat("DML Random Forest| Coef. Exprop:", round(dml_rf$coef, 4),    " | EE:", round(dml_rf$se, 4), "\n")

Warning message:
In meatHC(x, type = type, omega = omega) :
  HC0 covariances become (close to) singular if hat values are (close to) 1 as for observation(s) 11, 23, 54


DML Lasso CV     | Coef. Exprop: 0.8645  | EE: 0.3071 
DML Random Forest| Coef. Exprop: 0.8252  | EE: 0.259 


Variabilidad de $\hat\theta$ entre las $R$ repeticiones del sorteo de folds:

In [21]:
summary(dml_lasso$reps[, "coef"])
summary(dml_rf$reps[, "coef"])

   Min. 1st Qu.  Median    Mean 3rd Qu.    Max. 
 0.7755  0.7920  0.8252  0.8264  0.8565  0.8907 

### Resumen con DML

Note que con $n=64$ cada fold de entrenamiento tiene unas 51 observaciones, así que los estimadores de ML son ruidosos y el resultado puede ser sensible al sorteo de folds. Por eso conviene mirar la variabilidad entre repeticiones.

In [22]:
res_dml <- rbind(
  res,
  "DML Lasso CV"      = c(dml_lasso$coef, dml_lasso$se),
  "DML Random Forest" = c(dml_rf$coef,    dml_rf$se)
)
round(res_dml, 4)

                   Coef. Exprop     EE
IV base (latitud)        0.9692 0.2078
IV todos controles       0.9891 0.6786
IV controles Lasso       0.8349 0.3245
rlassoIV (hdm)           0.8349 0.2985
DML Lasso CV             0.8645 0.3071
DML Random Forest        0.8252 0.2590